# LeetCode 1379 — Find a Corresponding Node of a Binary Tree in a Clone of That Tree bằng Python

**Link đề:** https://leetcode.com/problems/find-a-corresponding-node-of-a-binary-tree-in-a-clone-of-that-tree/
**Độ khó:** Easy · **Chủ đề:** Tree, Depth-First Search, Breadth-First Search, Binary Tree

```python
# class TreeNode:
#     def __init__(self, x):
#         self.val = x
#         self.left = None
#         self.right = None

class Solution:
    def getTargetCopy(self, original: TreeNode, cloned: TreeNode, target: TreeNode) -> TreeNode:
        ...
```

### Mục tiêu học
1. Hiểu **cây nhị phân** (binary tree) trong Python: node là **đối tượng**, `left`/`right` là **tham chiếu** (reference).
2. Phân biệt `is` (cùng một đối tượng) và `==` (bằng giá trị) — mấu chốt của bài và của follow-up.
3. Tự dựng cây từ danh sách level-order kiểu LeetCode `[7,4,3,null,null,6,19]` và tự **clone** cây.
4. Đi qua 4 lời giải: tìm theo giá trị → **DFS song song** (đệ quy) → **BFS song song** (hàng đợi) → **ghi đường đi** rồi đi lại trên bản clone.
5. Trace, chứng minh, phân tích Time/Space, giới hạn đệ quy của Python, test và biến thể.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho hai cây nhị phân `original` và `cloned`, và một **tham chiếu** `target` tới một node **trong cây `original`**. Cây `cloned` là **bản sao** của `original`.

Trả về **tham chiếu tới node tương ứng trong `cloned`**.

Lưu ý: **không** được sửa hai cây hay node `target`; đáp án phải là một node **thuộc cây `cloned`**.

| tree (level-order) | target | Output |
|---|---|---|
| `[7,4,3,null,null,6,19]` | 3 | node 3 của cây clone |
| `[7]` | 7 | node 7 của cây clone |
| `[8,null,6,null,5,null,4,null,3,null,2,null,1]` | 4 | node 4 của cây clone |

**Constraints:** số node trong `[1, 10^4]`; giá trị các node **phân biệt**; `target` là một node của `original`, khác `None`.

**Follow up:** nếu cây có thể chứa **giá trị trùng nhau** thì giải thế nào?

## 2. Nền tảng: class, đối tượng và tham chiếu

LeetCode định nghĩa `TreeNode` bằng `class`. Mỗi lần gọi `TreeNode(x)` tạo **một đối tượng mới** trong bộ nhớ, có 3 thuộc tính: `val`, `left`, `right`. Biến trong Python chỉ là **cái tên trỏ tới** đối tượng.

In [1]:
class TreeNode:
    def __init__(self, x):
        self.val = x
        self.left = None
        self.right = None

    def __repr__(self):                      # cách node được in ra
        return f"TreeNode({self.val})"

a = TreeNode(7)
a.left = TreeNode(4)
a.right = TreeNode(3)
print(a, a.left, a.right, a.left.left)

TreeNode(7) TreeNode(4) TreeNode(3) None


`__init__` là **hàm khởi tạo**, chạy khi gọi `TreeNode(7)`; `self` là chính đối tượng đang tạo. `__repr__` quy định chuỗi hiển thị khi `print` — chỉ để dễ đọc, không ảnh hưởng thuật toán.

### 2.1. `is` khác `==`

- `x is y`: `x` và `y` **là cùng một đối tượng** (cùng địa chỉ bộ nhớ, xem bằng `id()`).
- `x == y`: so sánh **giá trị**. Với class tự viết không định nghĩa `__eq__`, `==` mặc định cũng so như `is`.

Hai node khác nhau có **cùng `val`** vẫn là **hai đối tượng khác nhau**.

In [2]:
p = TreeNode(3)
q = TreeNode(3)
r = p
print("p.val == q.val:", p.val == q.val)   # True: cùng giá trị
print("p is q:", p is q)                   # False: hai đối tượng
print("p is r:", p is r)                   # True: r chỉ là tên khác của p
print("id(p) == id(r):", id(p) == id(r), "| id(p) == id(q):", id(p) == id(q))

p.val == q.val: True
p is q: False
p is r: True
id(p) == id(r): True | id(p) == id(q): False


**Liên hệ với bài:** node 3 trong `original` và node 3 trong `cloned` có **cùng `val`** nhưng **không** phải cùng đối tượng. Đề yêu cầu trả đối tượng **thuộc cây clone** — trả `target` (đối tượng của original) là **sai** dù giá trị giống.

## 3. Công cụ: dựng cây từ list level-order và clone cây

LeetCode mô tả cây bằng list **theo tầng** (level-order), `null` là vị trí trống. Ta viết `build_tree` dùng **hàng đợi** `deque`:
- `deque` (trong `collections`) là hàng đợi hai đầu: `append` thêm cuối, `popleft` lấy đầu — đều `O(1)`.
- Lấy lần lượt từng node cha ra khỏi hàng đợi, gán con trái, con phải từ list.

In [3]:
from collections import deque

q = deque()
q.append(1); q.append(2); q.append(3)
print(q, "→ popleft:", q.popleft(), "→ còn", q)

deque([2, 3]) → popleft: 1 → còn deque([2, 3])


In [4]:
def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root

def to_list(root):
    # ngược lại build_tree: in cây thành list level-order (bỏ None thừa ở cuối)
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

null = None
t = build_tree([7, 4, 3, null, null, 6, 19])
print(to_list(t))
print(t, t.left, t.right, t.right.left, t.right.right)

[7, 4, 3, None, None, 6, 19]
TreeNode(7) TreeNode(4) TreeNode(3) TreeNode(6) TreeNode(19)


### 3.1. Clone cây (đệ quy) và hàm tìm node theo giá trị (để lấy `target`)

**Đệ quy** (recursion): hàm tự gọi lại chính nó trên bài toán nhỏ hơn. Clone một cây = tạo node gốc mới + clone cây con trái + clone cây con phải. Điều kiện dừng: cây rỗng (`None`) → trả `None`.

In [5]:
def clone_tree(root):
    if root is None:
        return None
    new = TreeNode(root.val)
    new.left = clone_tree(root.left)
    new.right = clone_tree(root.right)
    return new

def find_by_value(root, value):
    # chỉ dùng để LẤY target trong original khi chuẩn bị test
    stack = [root]
    while stack:
        node = stack.pop()
        if node is None:
            continue
        if node.val == value:
            return node
        stack.append(node.right)
        stack.append(node.left)
    return None

original = build_tree([7, 4, 3, null, null, 6, 19])
cloned = clone_tree(original)
target = find_by_value(original, 3)
print("Cây clone giống cây gốc về giá trị:", to_list(cloned) == to_list(original))
print("Nhưng là đối tượng khác:", cloned is not original, "| target thuộc original:", target is original.right)

Cây clone giống cây gốc về giá trị: True
Nhưng là đối tượng khác: True | target thuộc original: True


## 4. Cách 1 — Tìm theo giá trị trong cây clone

Vì đề đảm bảo **giá trị phân biệt**, cách trực giác: tìm node trong `cloned` có `val == target.val`.

Ta dùng **DFS bằng ngăn xếp** (stack): `list` với `append` (đẩy vào) và `pop()` (lấy phần tử **cuối** ra) — vào sau ra trước (LIFO).

In [6]:
def get_copy_by_value(original, cloned, target):
    stack = [cloned]
    while stack:
        node = stack.pop()
        if node is None:
            continue
        if node.val == target.val:
            return node
        stack.append(node.left)
        stack.append(node.right)
    return None

ans = get_copy_by_value(original, cloned, target)
print(ans, "| thuộc cây clone:", ans is cloned.right, "| không phải target:", ans is not target)

TreeNode(3) | thuộc cây clone: True | không phải target: True


**Nút thắt:** cách này **phụ thuộc giả thiết giá trị phân biệt**. Với follow-up (giá trị trùng), có thể trả nhầm node có cùng giá trị ở **vị trí khác**. Hơn nữa, nó **không dùng** cây `original` — bỏ phí thông tin "vị trí".

In [7]:
dup_original = build_tree([1, 1, 1])
dup_cloned = clone_tree(dup_original)
dup_target = dup_original.right              # node 1 bên PHẢI
wrong = get_copy_by_value(dup_original, dup_cloned, dup_target)
print("Trả về node bên phải của clone?", wrong is dup_cloned.right)
print("Thực tế trả node:", "gốc" if wrong is dup_cloned else "trái" if wrong is dup_cloned.left else "phải")

Trả về node bên phải của clone? False
Thực tế trả node: gốc


## 5. Insight — duyệt **hai cây cùng lúc**

Hai cây có **cùng hình dạng**. Nếu ta di chuyển **song song**: đi trái ở `original` thì cũng đi trái ở `cloned`, đi phải thì cùng đi phải — thì hai con trỏ `(o, c)` luôn chỉ tới **hai node ở cùng vị trí**.

Khi `o is target` (so sánh **đối tượng**, không so giá trị) → `c` chính là đáp án. Cách này đúng **cả khi giá trị trùng** (giải luôn follow-up), vì ta xác định node bằng **vị trí**, không bằng giá trị.

```text
original:  7          cloned:  7'
          / \                 / \
         4   3   ← target    4'  3'  ← đáp án
            / \                 / \
           6  19               6' 19'
```

## 6. Cách 2 — DFS song song bằng đệ quy

```python
def dfs(o, c):
    if o is None:          # nhánh rỗng: không có gì
        return None
    if o is target:        # tìm thấy: c là node tương ứng
        return c
    left = dfs(o.left, c.left)          # tìm bên trái
    if left is not None:
        return left
    return dfs(o.right, c.right)        # không có thì tìm bên phải
```

In [8]:
def get_copy_dfs(original, cloned, target):
    def dfs(o, c):
        if o is None:
            return None
        if o is target:
            return c
        left = dfs(o.left, c.left)
        if left is not None:
            return left
        return dfs(o.right, c.right)
    return dfs(original, cloned)

ans = get_copy_dfs(original, cloned, target)
print(ans, ans is cloned.right)

ans_dup = get_copy_dfs(dup_original, dup_cloned, dup_target)
print("Giá trị trùng — trả đúng node bên phải của clone?", ans_dup is dup_cloned.right)

TreeNode(3) True
Giá trị trùng — trả đúng node bên phải của clone? True


Hàm `dfs` định nghĩa **bên trong** `get_copy_dfs` (hàm lồng / nested function) nên nó "nhìn thấy" biến `target` của hàm ngoài — gọi là **closure**.

## 7. TRACE — DFS song song trên ví dụ 1 (target = 3)

In [9]:
def trace_get_copy_dfs(original, cloned, target):
    def dfs(o, c, depth):
        pad = "  " * depth
        if o is None:
            print(f"{pad}(o=None) → nhánh rỗng, trả None")
            return None
        same = o is target
        print(f"{pad}thăm o={o.val} (id …{id(o) % 10000:04d}), c={c.val} (id …{id(c) % 10000:04d}) | o is target? {same}")
        if same:
            print(f"{pad}→ TÌM THẤY, trả node clone {c}")
            return c
        left = dfs(o.left, c.left, depth + 1)
        if left is not None:
            return left
        return dfs(o.right, c.right, depth + 1)
    return dfs(original, cloned, 0)

res = trace_get_copy_dfs(original, cloned, target)
print("Kết quả:", res, "| thuộc clone:", res is cloned.right)

thăm o=7 (id …0768), c=7 (id …3824) | o is target? False
  thăm o=4 (id …6048), c=4 (id …3440) | o is target? False
    (o=None) → nhánh rỗng, trả None
    (o=None) → nhánh rỗng, trả None
  thăm o=3 (id …4128), c=3 (id …3440) | o is target? True
  → TÌM THẤY, trả node clone TreeNode(3)
Kết quả: TreeNode(3) | thuộc clone: True


Đọc trace: bắt đầu ở cặp gốc `(7, 7')`; đi trái `(4, 4')` → hai con của 4 đều rỗng → quay lại; đi phải `(3, 3')`: `o is target` đúng → trả `3'`. Thụt lề thể hiện **độ sâu** đệ quy. `id` khác nhau giữa `o` và `c` chứng tỏ chúng là hai đối tượng khác nhau dù cùng giá trị.

Bảng tóm tắt:

| Bước | o | c | o is target? | Hành động |
|---|---|---|---|---|
| 1 | 7 | 7' | không | đi trái |
| 2 | 4 | 4' | không | đi trái |
| 3 | None | | | trả None, đi phải của 4 |
| 4 | None | | | trả None → 4 trả None |
| 5 | 3 | 3' | **có** | trả 3' |

### 7.1. Giới hạn đệ quy của Python

Python giới hạn độ sâu đệ quy (mặc định khoảng 1000) để tránh tràn ngăn xếp. Cây có tới `10^4` node có thể là **một chuỗi lệch** (như ví dụ 3) → độ sâu `10^4` → đệ quy có thể báo `RecursionError`. Ta kiểm chứng bằng `sys.getrecursionlimit()` và một cây lệch dài.

In [10]:
import sys
print("Giới hạn đệ quy hiện tại:", sys.getrecursionlimit())

def build_chain(n):
    # cây lệch phải: 1 → 2 → ... → n
    root = TreeNode(1)
    cur = root
    for v in range(2, n + 1):
        cur.right = TreeNode(v)
        cur = cur.right
    return root, cur          # trả gốc và node cuối (sâu nhất)

chain, deepest = build_chain(10_000)
try:
    get_copy_dfs(chain, chain, deepest)     # dùng chính nó làm "clone" chỉ để thử độ sâu
    print("Đệ quy chạy được")
except RecursionError as e:
    print("RecursionError:", str(e)[:60])

Giới hạn đệ quy hiện tại: 1000
RecursionError: maximum recursion depth exceeded


`try / except` bắt lỗi để chương trình không dừng. Vì vậy với cây sâu, nên dùng **lặp** (stack/queue tự quản lý) — đó là hai cách tiếp theo. (Một số nơi tăng giới hạn bằng `sys.setrecursionlimit`, nhưng lặp an toàn hơn.)

## 8. Cách 3 — BFS song song bằng hàng đợi (lời giải chính)

Đưa **cặp** `(o, c)` vào hàng đợi; mỗi lần lấy một cặp ra, nếu `o is target` thì trả `c`, ngược lại đẩy cặp con trái và cặp con phải (nếu có).

In [11]:
def get_copy_bfs(original, cloned, target):
    queue = deque([(original, cloned)])
    while queue:
        o, c = queue.popleft()
        if o is target:
            return c
        if o.left:
            queue.append((o.left, c.left))
        if o.right:
            queue.append((o.right, c.right))
    return None

print(get_copy_bfs(original, cloned, target) is cloned.right)
print("BFS trên cây lệch 10^4 node:", get_copy_bfs(chain, chain, deepest) is deepest)

True
BFS trên cây lệch 10^4 node: True


`if o.left:` — một đối tượng `TreeNode` được coi là `True`, `None` là `False`, nên đây là cách viết gọn của `if o.left is not None`.

### 8.1. Trace BFS trên ví dụ 1 với target = 6

In [12]:
def trace_get_copy_bfs(original, cloned, target):
    queue = deque([(original, cloned)])
    step = 1
    while queue:
        print(f"Bước {step}: hàng đợi = {[o.val for o, _ in queue]}")
        o, c = queue.popleft()
        if o is target:
            print(f"  lấy ({o.val}, {c.val}'): o is target → trả {c.val}'")
            return c
        print(f"  lấy ({o.val}, {c.val}'): chưa phải, thêm các cặp con")
        if o.left:
            queue.append((o.left, c.left))
        if o.right:
            queue.append((o.right, c.right))
        step += 1
    return None

t6 = find_by_value(original, 6)
r6 = trace_get_copy_bfs(original, cloned, t6)
print("Đúng node clone?", r6 is cloned.right.left)

Bước 1: hàng đợi = [7]
  lấy (7, 7'): chưa phải, thêm các cặp con
Bước 2: hàng đợi = [4, 3]
  lấy (4, 4'): chưa phải, thêm các cặp con
Bước 3: hàng đợi = [3]
  lấy (3, 3'): chưa phải, thêm các cặp con
Bước 4: hàng đợi = [6, 19]
  lấy (6, 6'): o is target → trả 6'
Đúng node clone? True


## 9. Cách 4 — Ghi đường đi trong `original`, đi lại trong `cloned`

Một góc nhìn khác: vị trí của một node trong cây xác định duy nhất bởi **đường đi từ gốc** (chuỗi `L`/`R`). Bước 1: tìm đường đi tới `target` trong `original`. Bước 2: đi đúng đường đó trên `cloned`.

Cách này **tách** hai việc "tìm" và "đi", hữu ích khi phải tra nhiều cây clone cho cùng một target.

In [13]:
def path_to(root, target):
    # DFS lặp, lưu kèm đường đi; trả list như ['R', 'L']
    stack = [(root, [])]
    while stack:
        node, path = stack.pop()
        if node is target:
            return path
        if node.right:
            stack.append((node.right, path + ["R"]))
        if node.left:
            stack.append((node.left, path + ["L"]))
    return None

def follow(root, path):
    node = root
    for step in path:
        node = node.left if step == "L" else node.right
    return node

def get_copy_path(original, cloned, target):
    return follow(cloned, path_to(original, target))

print("Đường tới 6:", path_to(original, t6))
print(get_copy_path(original, cloned, t6) is cloned.right.left)

Đường tới 6: ['R', 'L']
True


Chú ý `path + ["R"]` tạo **list mới** cho mỗi node (để các nhánh không dùng chung một list). Cách này tốn thêm bộ nhớ `O(h)` cho mỗi phần tử trong stack — dùng để minh hoạ ý tưởng; muốn tiết kiệm có thể lưu con trỏ cha.

## 10. Code nộp LeetCode

Bản BFS song song (lặp, an toàn với cây sâu, giải luôn follow-up):

In [14]:
from collections import deque

class Solution:
    def getTargetCopy(self, original: TreeNode, cloned: TreeNode, target: TreeNode) -> TreeNode:
        queue = deque([(original, cloned)])
        while queue:
            o, c = queue.popleft()
            if o is target:
                return c
            if o.left:
                queue.append((o.left, c.left))
            if o.right:
                queue.append((o.right, c.right))

sol = Solution()
for values, tv in [([7, 4, 3, None, None, 6, 19], 3), ([7], 7),
                   ([8, None, 6, None, 5, None, 4, None, 3, None, 2, None, 1], 4)]:
    o = build_tree(values)
    c = clone_tree(o)
    tg = find_by_value(o, tv)
    ans = sol.getTargetCopy(o, c, tg)
    print(values, "target", tv, "→", ans, "| là node clone:", ans is find_by_value(c, tv), "| khác target:", ans is not tg)

[7, 4, 3, None, None, 6, 19] target 3 → TreeNode(3) | là node clone: True | khác target: True
[7] target 7 → TreeNode(7) | là node clone: True | khác target: True
[8, None, 6, None, 5, None, 4, None, 3, None, 2, None, 1] target 4 → TreeNode(4) | là node clone: True | khác target: True


Bản DFS lặp bằng stack (cũng Accepted):

In [15]:
class SolutionStack:
    def getTargetCopy(self, original: TreeNode, cloned: TreeNode, target: TreeNode) -> TreeNode:
        stack = [(original, cloned)]
        while stack:
            o, c = stack.pop()
            if o is target:
                return c
            if o.right:
                stack.append((o.right, c.right))
            if o.left:
                stack.append((o.left, c.left))

print(SolutionStack().getTargetCopy(original, cloned, target) is cloned.right)

True


## 11. Vì sao đúng? Độ phức tạp

**Invariant:** mọi cặp `(o, c)` nằm trong hàng đợi/stack đều là **hai node ở cùng vị trí** (cùng đường đi từ gốc) trong `original` và `cloned`.
- Ban đầu `(original, cloned)` — hai gốc, đường đi rỗng. ✓
- Nếu `(o, c)` cùng vị trí với đường đi `P` thì `(o.left, c.left)` cùng vị trí `P + L`, `(o.right, c.right)` cùng vị trí `P + R`. Vì hai cây cùng hình dạng, `o.left` tồn tại ⇔ `c.left` tồn tại. ✓

BFS/DFS duyệt **mọi** node của `original` (cho tới khi gặp `target`), và `target` thuộc `original` → chắc chắn gặp. Khi gặp, theo invariant `c` cùng vị trí với `target` → đúng node cần tìm. So sánh bằng `is` nên không bị nhầm giá trị trùng.

Gọi `N` = số node, `H` = chiều cao cây, `W` = độ rộng lớn nhất của một tầng.

| Cách | Thời gian | Bộ nhớ phụ | Giá trị trùng? |
|---|---|---|---|
| 1. Tìm theo giá trị | `O(N)` | `O(H)` (stack) | ✗ sai |
| 2. DFS đệ quy song song | `O(N)` | `O(H)` call stack | ✓ (nhưng có thể vượt giới hạn đệ quy) |
| 3. BFS song song | `O(N)` | `O(W)` | ✓ |
| 3'. DFS lặp song song | `O(N)` | `O(H)` | ✓ |
| 4. Ghi đường đi | `O(N)` + `O(H)` | `O(H²)` với list path sao chép | ✓ |

## 12. Bộ test: mẫu đề, biên, giá trị trùng, ngẫu nhiên

In [16]:
import random

def all_nodes(root):
    out, stack = [], [root]
    while stack:
        n = stack.pop()
        if n:
            out.append(n)
            stack.append(n.right)
            stack.append(n.left)
    return out

def check_all_targets(original, cloned, solver):
    # với MỌI node của original làm target, đáp án phải là node clone cùng vị trí
    o_nodes = all_nodes(original)
    c_nodes = all_nodes(cloned)          # cùng thứ tự duyệt → cùng vị trí
    for o, c in zip(o_nodes, c_nodes):
        ans = solver(original, cloned, o)
        assert ans is c, (o.val, ans)
    return len(o_nodes)

solvers = [get_copy_dfs, get_copy_bfs, get_copy_path,
           Solution().getTargetCopy, SolutionStack().getTargetCopy]
for values in [[7, 4, 3, None, None, 6, 19], [7], [8, None, 6, None, 5, None, 4, None, 3, None, 2, None, 1],
               [1, 2, 3, 4, 5, 6, 7], [1, 1, 1, 1, None, 1]]:
    o = build_tree(values)
    c = clone_tree(o)
    for s in solvers:
        k = check_all_targets(o, c, s)
    print(f"PASS {values}: {k} target × {len(solvers)} cách")

PASS [7, 4, 3, None, None, 6, 19]: 5 target × 5 cách
PASS [7]: 1 target × 5 cách
PASS [8, None, 6, None, 5, None, 4, None, 3, None, 2, None, 1]: 7 target × 5 cách
PASS [1, 2, 3, 4, 5, 6, 7]: 7 target × 5 cách
PASS [1, 1, 1, 1, None, 1]: 5 target × 5 cách


`zip(a, b)` ghép từng cặp phần tử cùng vị trí của hai list: `zip([1,2],[3,4])` → `(1,3), (2,4)`. Vì `all_nodes` duyệt hai cây cùng hình dạng theo cùng thứ tự, các cặp trong `zip` là các node tương ứng.

In [17]:
def random_tree(rng, n, max_val):
    # chèn n node vào vị trí trống ngẫu nhiên
    root = TreeNode(rng.randint(1, max_val))
    nodes = [root]
    for _ in range(n - 1):
        while True:
            parent = rng.choice(nodes)
            side = rng.choice(["left", "right"])
            if getattr(parent, side) is None:
                child = TreeNode(rng.randint(1, max_val))
                setattr(parent, side, child)
                nodes.append(child)
                break
    return root

rng = random.Random(1379)
for _ in range(150):
    o = random_tree(rng, rng.randint(1, 40), max_val=3)       # max_val nhỏ → nhiều giá trị trùng
    c = clone_tree(o)
    for s in solvers:
        check_all_targets(o, c, s)
print("150 cây ngẫu nhiên (nhiều giá trị trùng) × mọi target × 5 cách: PASS")

deep_clone = build_chain(10_000)[0]
assert Solution().getTargetCopy(chain, deep_clone, deepest).val == 10_000
assert Solution().getTargetCopy(chain, deep_clone, deepest) is not deepest
print("Cây lệch 10^4 node: bản lặp chạy tốt — PASS")

150 cây ngẫu nhiên (nhiều giá trị trùng) × mọi target × 5 cách: PASS
Cây lệch 10^4 node: bản lặp chạy tốt — PASS


`getattr(obj, "left")` đọc thuộc tính theo **tên dạng chuỗi**; `setattr(obj, "left", v)` gán. Tiện khi tên thuộc tính được chọn ngẫu nhiên.

Ở test cây lệch, `deep_clone` được dựng **riêng** bằng `build_chain` (hai cây cùng hình dạng) — không dùng `clone_tree` đệ quy để tránh `RecursionError`.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Trả về chính `target`
`return target` — đúng giá trị nhưng **không thuộc** cây clone. LeetCode kiểm tra bằng đối tượng → Wrong Answer.

In [18]:
def wrong_return_target(original, cloned, target):
    return target

ans = wrong_return_target(original, cloned, target)
print("Cùng giá trị:", ans.val == cloned.right.val, "| Là node clone:", ans is cloned.right)

Cùng giá trị: True | Là node clone: False


### Lỗi 2: So sánh bằng `val` khi duyệt song song
`if o.val == target.val` — trả node đầu tiên có cùng giá trị, sai khi giá trị trùng (follow-up). Phải dùng `o is target`.

In [19]:
def wrong_compare_val(original, cloned, target):
    queue = deque([(original, cloned)])
    while queue:
        o, c = queue.popleft()
        if o.val == target.val:          # SAI với giá trị trùng
            return c
        if o.left: queue.append((o.left, c.left))
        if o.right: queue.append((o.right, c.right))

ans = wrong_compare_val(dup_original, dup_cloned, dup_target)
print("Trả đúng node bên phải?", ans is dup_cloned.right, "| thực tế là gốc?", ans is dup_cloned)

Trả đúng node bên phải? False | thực tế là gốc? True


### Lỗi 3: Duyệt hai cây **lệch nhịp**
Đẩy `(o.left, c.right)` nhầm chiều → hai con trỏ không còn cùng vị trí.

### Lỗi 4: Quên trả kết quả từ nhánh trái trong đệ quy
```python
def dfs(o, c):
    if o is None: return None
    if o is target: return c
    dfs(o.left, c.left)          # SAI: bỏ mất kết quả
    return dfs(o.right, c.right)
```

In [20]:
def wrong_lost_result(original, cloned, target):
    def dfs(o, c):
        if o is None:
            return None
        if o is target:
            return c
        dfs(o.left, c.left)                 # kết quả bị bỏ đi
        return dfs(o.right, c.right)
    return dfs(original, cloned)

t4 = find_by_value(original, 4)              # target nằm bên TRÁI
print("Sai :", wrong_lost_result(original, cloned, t4), "| Đúng:", get_copy_dfs(original, cloned, t4))

Sai : None | Đúng: TreeNode(4)


### Mẹo nhận dạng pattern

- **Hai cây cùng hình dạng** (so sánh, gộp, ánh xạ node) → duyệt **song song** bằng cặp `(node1, node2)`.
- Cần xác định node **theo danh tính** (identity) chứ không theo giá trị → dùng `is`.
- Cây có thể sâu tới `10^4` → ưu tiên **lặp** (stack/queue) trong Python.
- Bài anh em: *Same Tree* (100), *Symmetric Tree* (101), *Merge Two Binary Trees* (617), *Clone Graph* (133), *Leaf-Similar Trees* (872).

## 14. Biến thể

### 14.1. Same Tree (LeetCode 100) — cũng duyệt song song

In [21]:
def is_same_tree(p, q):
    stack = [(p, q)]
    while stack:
        a, b2 = stack.pop()
        if a is None and b2 is None:
            continue
        if a is None or b2 is None or a.val != b2.val:
            return False
        stack.append((a.left, b2.left))
        stack.append((a.right, b2.right))
    return True

print(is_same_tree(original, cloned), is_same_tree(original, build_tree([7, 4, 3, None, None, 6])))

True False


### 14.2. Ánh xạ toàn bộ: dict từ node gốc sang node clone
Nếu phải trả lời **rất nhiều** truy vấn target, dựng một lần dict `{id(o): c}` rồi tra `O(1)` mỗi truy vấn.

In [22]:
def build_mapping(original, cloned):
    mapping = {}
    stack = [(original, cloned)]
    while stack:
        o, c = stack.pop()
        mapping[id(o)] = c
        if o.left:
            stack.append((o.left, c.left))
        if o.right:
            stack.append((o.right, c.right))
    return mapping

mp = build_mapping(original, cloned)
print(len(mp), "cặp; tra node 19:", mp[id(find_by_value(original, 19))], mp[id(find_by_value(original, 19))] is cloned.right.right)

5 cặp; tra node 19: TreeNode(19) True


Dùng `id(o)` làm key để khoá theo **danh tính** đối tượng (an toàn kể cả khi giá trị trùng). Lưu ý: chỉ dùng `id` khi các đối tượng còn sống (chưa bị xoá), vì id có thể được tái sử dụng.

### 14.3. Mirror clone: cây clone là **ảnh gương** của original
Duyệt song song nhưng **đổi chiều**: con trái của `o` tương ứng con phải của `c`.

In [23]:
def mirror(root):
    if root is None:
        return None
    m = TreeNode(root.val)
    m.left, m.right = mirror(root.right), mirror(root.left)
    return m

def get_copy_in_mirror(original, mirrored, target):
    stack = [(original, mirrored)]
    while stack:
        o, c = stack.pop()
        if o is target:
            return c
        if o.left:
            stack.append((o.left, c.right))
        if o.right:
            stack.append((o.right, c.left))

mir = mirror(original)
print(to_list(mir), "| node tương ứng của 6:", get_copy_in_mirror(original, mir, t6), get_copy_in_mirror(original, mir, t6) is mir.left.right)

[7, 3, 4, 19, 6] | node tương ứng của 6: TreeNode(6) True


## 15. Bài tập (TODO — đáp án ở cuối)

### Bài 1 — Tính tay
Với cây `[1,2,3,4,5,null,6]` và target là node `5`, liệt kê thứ tự các cặp `(o, c)` mà **BFS song song** lấy ra khỏi hàng đợi cho tới khi tìm thấy.

### Bài 2 — DFS song song bằng stack

In [24]:
def exercise_get_copy_stack(original, cloned, target):
    # TODO: stack các cặp (o, c), so sánh bằng `is`
    pass

# Gợi ý test (sau khi làm xong):
# o = build_tree([1, 1, 1, 1, None, 1]); c = clone_tree(o)
# assert check_all_targets(o, c, exercise_get_copy_stack) == 5

### Bài 3 — Đường đi tới target
Viết `exercise_path(root, target)` trả chuỗi như `"RL"` (đi phải rồi trái); gốc trả `""`.

In [25]:
def exercise_path(root, target):
    # TODO
    pass

# Gợi ý test:
# assert exercise_path(original, t6) == "RL"
# assert exercise_path(original, original) == ""

### Bài 4 — Độ sâu của target
Trả về độ sâu (gốc có độ sâu 0) của `target` bằng BFS theo tầng.

In [26]:
def exercise_depth(root, target):
    # TODO
    pass

# Gợi ý test:
# assert exercise_depth(original, t6) == 2
# assert exercise_depth(original, original) == 0

### Bài 5 — Clone cây bằng vòng lặp (không đệ quy)
Viết `exercise_clone_iterative(root)` an toàn với cây sâu `10^4`.

In [27]:
def exercise_clone_iterative(root):
    # TODO: stack các cặp (node gốc, node mới)
    pass

# Gợi ý test:
# assert to_list(exercise_clone_iterative(original)) == to_list(original)

### Bài 6 — Mini challenge: nhiều target
Cho list `targets` (các node của `original`), trả list node tương ứng trong `cloned`, tổng thời gian `O(N + len(targets))`.

In [28]:
def exercise_many_targets(original, cloned, targets):
    # TODO
    pass

# Gợi ý test:
# ts = all_nodes(original)
# assert all(a is b for a, b in zip(exercise_many_targets(original, cloned, ts), all_nodes(cloned)))

## 16. Gợi ý 3 mức

**Bài 1** — (1) BFS lấy theo tầng, trái trước phải. (2) Tầng 0: 1; tầng 1: 2, 3; tầng 2: 4, 5, 6. (3) Dừng khi lấy ra cặp chứa 5.

**Bài 2** — (1) Giống `SolutionStack`. (2) `stack.pop()` lấy cặp, `is target` thì trả. (3) Đẩy cặp con cùng chiều.

**Bài 3** — (1) DFS kèm đường đi. (2) Lưu `(node, path_string)` trong stack. (3) `path + "L"` khi đi trái.

**Bài 4** — (1) BFS kèm độ sâu. (2) Hàng đợi chứa `(node, depth)`. (3) Con có `depth + 1`.

**Bài 5** — (1) Tạo gốc mới trước. (2) Stack các cặp `(old, new)`; với mỗi con của `old`, tạo con mới cho `new`. (3) Đẩy cặp con vào stack.

**Bài 6** — (1) Mục 15.2. (2) Dựng dict `id(o) → c` một lần. (3) `[mapping[id(t)] for t in targets]`.

## 17. Đáp án tham khảo

**Bài 1:** hàng đợi lấy ra lần lượt `(1,1')`, `(2,2')`, `(3,3')`, `(4,4')`, `(5,5')` → trả `5'`.

In [29]:
def answer_get_copy_stack(original, cloned, target):
    stack = [(original, cloned)]
    while stack:
        o, c = stack.pop()
        if o is target:
            return c
        if o.right:
            stack.append((o.right, c.right))
        if o.left:
            stack.append((o.left, c.left))
    return None

def answer_path(root, target):
    stack = [(root, "")]
    while stack:
        node, path = stack.pop()
        if node is target:
            return path
        if node.right:
            stack.append((node.right, path + "R"))
        if node.left:
            stack.append((node.left, path + "L"))
    return None

def answer_depth(root, target):
    queue = deque([(root, 0)])
    while queue:
        node, d = queue.popleft()
        if node is target:
            return d
        if node.left:
            queue.append((node.left, d + 1))
        if node.right:
            queue.append((node.right, d + 1))
    return -1

def answer_clone_iterative(root):
    if root is None:
        return None
    new_root = TreeNode(root.val)
    stack = [(root, new_root)]
    while stack:
        old, new = stack.pop()
        if old.left:
            new.left = TreeNode(old.left.val)
            stack.append((old.left, new.left))
        if old.right:
            new.right = TreeNode(old.right.val)
            stack.append((old.right, new.right))
    return new_root

def answer_many_targets(original, cloned, targets):
    mapping = build_mapping(original, cloned)
    return [mapping[id(t)] for t in targets]

# Bài 1
ex = build_tree([1, 2, 3, 4, 5, None, 6]); exc = clone_tree(ex)
assert trace_get_copy_bfs(ex, exc, find_by_value(ex, 5)) is exc.left.right

o = build_tree([1, 1, 1, 1, None, 1]); c = clone_tree(o)
assert check_all_targets(o, c, answer_get_copy_stack) == 5
assert answer_path(original, t6) == "RL" and answer_path(original, original) == ""
assert answer_depth(original, t6) == 2 and answer_depth(original, original) == 0
assert to_list(answer_clone_iterative(original)) == to_list(original)
deep_copy = answer_clone_iterative(chain)
assert answer_depth(deep_copy, follow(deep_copy, "R" * 9999)) == 9999
ts = all_nodes(original)
assert all(a is b for a, b in zip(answer_many_targets(original, cloned, ts), all_nodes(cloned)))
print("Đáp án bài tập: PASS toàn bộ")

Bước 1: hàng đợi = [1]
  lấy (1, 1'): chưa phải, thêm các cặp con
Bước 2: hàng đợi = [2, 3]
  lấy (2, 2'): chưa phải, thêm các cặp con
Bước 3: hàng đợi = [3, 4, 5]
  lấy (3, 3'): chưa phải, thêm các cặp con
Bước 4: hàng đợi = [4, 5, 6]
  lấy (4, 4'): chưa phải, thêm các cặp con
Bước 5: hàng đợi = [5, 6]
  lấy (5, 5'): o is target → trả 5'
Đáp án bài tập: PASS toàn bộ


## 18. Tổng kết — template nhớ nhanh + checklist

```python
queue = deque([(original, cloned)])        # duyệt SONG SONG hai cây
while queue:
    o, c = queue.popleft()
    if o is target:                        # so sánh ĐỐI TƯỢNG, không so val
        return c
    if o.left:  queue.append((o.left,  c.left))
    if o.right: queue.append((o.right, c.right))
```

**Checklist trước khi submit**
- [ ] Trả node **thuộc cây clone**, không trả `target`.
- [ ] So sánh bằng `is` (đúng cả khi giá trị trùng — follow-up).
- [ ] Hai con trỏ luôn đi **cùng chiều**.
- [ ] Dùng lặp để an toàn với cây sâu `10^4`.
- [ ] Đã test cây 1 node, cây lệch, cây có giá trị trùng.